# Capstone C — Wavely Playlists

**Level:** advanced — nested JSON and changes over time

Every night Wavely exports a **snapshot** of all user playlists: which tracks are in each playlist on that day.
The recommendations team asks:
1. Which tracks were **added** to playlists most often during the week? Which were **removed** most?
2. How do playlist sizes change day by day?
3. Which **genres** dominate public playlists, and do collaborators (people other than the owner) add different music?

Snapshots only tell you the *state* of each playlist per day. **Additions and removals are not in the data** — you have to
derive them by comparing consecutive snapshots. That's the challenge.

## The data (`data/capstone/playlists/`)
`playlist_snapshots_2026-09-01.jsonl` … `playlist_snapshots_2026-09-07.jsonl`, one JSON object per playlist per day:

```json
{"snapshot_date": "2026-09-01",
 "playlist": {"id": "P0001", "name": "Focus 1", "is_public": true, "owner": {"user_id": "U0123", "plan": "premium"}},
 "tracks": [{"track_id": "T0042", "added_at": "2026-08-14T10:00:00", "added_by": "U0123"}, ...]}
```
Use `data/reference/tracks.csv` and `artists.csv` for titles, genres and artist names.

## Requirements
- [ ] **Bronze:** the raw snapshots (keep them nested) + `_source_file` + `_ingested_at`
- [ ] **Silver:** one row per (snapshot_date, playlist, track) — flattened with `explode`; invalid rows to **quarantine**
- [ ] **Gold:** at least 2 tables, including daily **additions and removals** derived from consecutive snapshots
- [ ] **Quality checks:** at least 3, including the reconciliation *size(today) = size(yesterday) + added − removed*
- [ ] **`run_pipeline()`** runs everything; show that running it twice gives the same numbers
- [ ] A chart or table with your answer, and a `README.md` on GitHub
- [ ] **Stretch:** make it incremental — process one new snapshot per run (session 9)

The structure below is a suggestion. Change it as you like.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
SRC = DATA / "capstone" / "playlists"
LAKE = DATA.parent / "capstone_lake" / "playlists"
KEY = ["snapshot_date", "playlist_id", "track_id"]


def check(name, bad_rows):
    n_bad = bad_rows.count()
    print(f"{'PASS' if n_bad == 0 else 'FAIL'}  {name}  ({n_bad} bad rows)")
    return n_bad == 0


def read_ref(name):
    return spark.read.csv(str(DATA / "reference" / f"{name}.csv"), header=True, inferSchema=True)

## 1. Explore

In [ ]:
# YOUR CODE HERE

## 2. Bronze

In [ ]:
def build_bronze():
    # YOUR CODE HERE
    ...

## 3. Silver
Decide your rules. Write them down here:
- ...

In [ ]:
def build_silver():
    # YOUR CODE HERE
    ...

## 4. Gold
Hint for additions: a track is *added* on day *d* if it is in the snapshot of day *d* but **not** in day *d − 1*.
Shift yesterday's snapshot forward by one day (`F.date_add`) and use a `left_anti` join.

In [ ]:
def build_gold():
    # YOUR CODE HERE
    ...

## 5. Quality checks

In [ ]:
def run_checks():
    # YOUR CODE HERE
    ...

## 6. Run it (twice)

In [ ]:
def run_pipeline():
    build_bronze()
    build_silver()
    build_gold()
    run_checks()
    return spark.read.parquet(str(LAKE / "gold" / "track_changes")).count()


print("Idempotent:", run_pipeline() == run_pipeline())

## 7. Answers

In [ ]:
# YOUR CODE HERE